# 02 — Data Preparation

**Input:** `data/processed/hour_clean.csv`

**Goals:**
- Feature engineering from datetime and existing columns
- Encode categorical variables (label, one-hot)
- Scale / normalise numerical features
- Time-aware train/test split (no leakage)
- Save `X_train`, `X_test`, `y_train`, `y_test`

**Key concept:** always split on time first, then fit transformers only on train — never fit on test.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib, os

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
df = pd.read_csv("../data/processed/hour_clean.csv", parse_dates=["dteday"])
df.head()

## 1. Feature engineering

Ideas to try:
- `is_weekend` from `weekday`
- `hour_sin` / `hour_cos` — cyclical encoding of hour
- `month_sin` / `month_cos` — cyclical encoding of month
- `temp_hum_interaction` — temp × (1 − hum)
- `lag_1h`, `lag_24h` — lagged target (careful: only valid for ML, not forecasting notebook)

In [ ]:
# Cyclical encoding helper
def cyclical(series, period):
    return np.sin(2 * np.pi * series / period), np.cos(2 * np.pi * series / period)

# TODO: add your features here
df["is_weekend"] = (df["weekday"].isin([0, 6])).astype(int)
df["hour_sin"], df["hour_cos"] = cyclical(df["hr"], 24)
df["month_sin"], df["month_cos"] = cyclical(df["mnth"], 12)

# YOUR CODE — add more features

print(df.shape)
df.head(3)

## 2. Train / test split (time-based)

In [ ]:
# Use last ~3 months as test set (time-aware split)
cutoff = df["dteday"].max() - pd.DateOffset(months=3)
train = df[df["dteday"] <= cutoff].copy()
test  = df[df["dteday"] >  cutoff].copy()
print(f"Train: {len(train)} rows ({train['dteday'].min().date()} – {train['dteday'].max().date()})")
print(f"Test:  {len(test)} rows  ({test['dteday'].min().date()} – {test['dteday'].max().date()})")

## 3. Select features & target

In [ ]:
# Drop columns not used as features
drop_cols = ["dteday", "cnt", "casual", "registered"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X_train = train[feature_cols]
X_test  = test[feature_cols]
y_train = train["cnt"]
y_test  = test["cnt"]

print("Features:", feature_cols)

## 4. Scaling

Tree-based models don't need scaling, but linear models do. Fit scaler on train only.

In [ ]:
# TODO: identify which columns need scaling vs which are already bounded/categorical
scale_cols = ["temp", "atemp", "hum", "windspeed"]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])  # transform only!

print("Scaler fitted on train, applied to test.")

## 5. Save

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
joblib.dump(scaler, "../data/processed/scaler.pkl")
print("Saved train/test splits and scaler.")